# YZ50 — Week 8: Building a GPT

- **Character-level language modeling** represents text as character IDs and learns to predict the **next character** from the available context.

- A **bigram language model** provides a baseline before attention. It predicts from the current character alone, making the role of a longer context easier to understand.

- **Self-attention** replaces uniform averaging with learned communication. Queries and keys determine which tokens are relevant; values determine what information is gathered.

- **Multi-head attention** uses several smaller heads in parallel. Their outputs are concatenated and projected back into the shared representation.

- **Feedforward networks, residual connections, and LayerNorm** extend this communication mechanism into a Transformer block that can be stacked to build a GPT.

In this week, I move from a single attention head toward **stacked, causal Transformer blocks**, then examine model size, dropout, and Turkish character-level modeling through measured validation losses and generated text. The default study run uses 200 Transformer updates; `study_run = False` selects the 5,000-update budget.

In [1]:
from pathlib import Path
import time
from collections import Counter
import torch
import torch.nn as nn
from torch.nn import functional as F
from IPython.display import display, Markdown

study_run = True
seed = 1337
torch.manual_seed(seed)
torch.set_num_threads(2)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
data_dir = Path.cwd()
if not (data_dir / 'turkish_input.txt').exists():
    data_dir = data_dir / 'week-8'
print('PyTorch:', torch.__version__, 'device:', device)

PyTorch: 2.14.0+cu126 device: cuda


## Read Tiny Shakespeare

A language model receives numerical tokens, so I first inspect the **raw text** before constructing the tokenizer.

The following cells read the local UTF-8 file, print its length, and show the first 1,000 characters. Newlines, spaces, speaker names, and punctuation all belong to the prediction problem. They are preserved rather than removed during preprocessing.

In [2]:
# read it in to inspect it
with open(data_dir / 'input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [4]:
# let's look at the first 1000 characters
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



### Write the Character Tokenizer

**Tokenization** converts raw text into units that can be mapped to numerical IDs. Here, each token is **one character**, rather than a word or subword.

`sorted(set(text))` identifies the vocabulary and gives a consistent ordering for the same corpus. `stoi` and `itos` implement the two directions:

- **Encode:** text → token IDs.
- **Decode:** token IDs → text.

`encode("hii there")` returns integers; `decode` reconstructs the string. The round-trip assertion checks that the mapping preserves the original text.

The corpus becomes a one-dimensional `torch.long` tensor with shape `(N,)`, where **N** is the number of characters. These integers select embedding rows; their numerical distance does not express character similarity. The tokenizer has no unknown-character token: it represents the characters observed in this corpus.

In [5]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [6]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [7]:
assert decode(encode(text[:100])) == text[:100]

In [8]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000]) # the 1000 characters we looked at earier will to the GPT look like this

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

### Split the Data and Sample Context Windows

The first **90%** of the character tensor becomes `train_data`; the remaining **10%** becomes `val_data`. This chronological split keeps validation sequences outside the region used for gradient updates.

`block_size` is the maximum input-window length. A window beginning at position \(i\) gives:

\[
x=\text{data}[i:i+T],\qquad y=\text{data}[i+1:i+T+1].
\]

The target at every position is the next character. Eight input characters therefore give eight predictions, not just a prediction after the final character.

`get_batch` samples **B independent starting positions**, slices a window and its shifted targets at each position, and stacks them into `(B,T)` tensors. With `B=4` and `T=8`, the batch contains 32 next-character targets.

The printed prefix `x[:t+1]` is the context available for predicting `y[t]`. The assertion `xb[:,1:] == yb[:,:-1]` checks target alignment. Later, causal masking prevents a prediction from using the remaining future positions within its input window.

In [9]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [10]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [11]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [12]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

In [13]:
print(xb) # our input to the transformer

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [14]:
assert torch.equal(xb[:,1:], yb[:,:-1])

## Train a Bigram Baseline

A **bigram model** predicts the next character using only the current character. It does not combine earlier context, even though a batch contains several positions.

`nn.Embedding(vocab_size, vocab_size)` is a learned lookup table. Each character selects a row of **V logits**, one score for every possible next character:

**input IDs `(B,T)` → lookup table → logits `(B,T,V)`**

Logits are unnormalized scores. Cross-entropy applies log-softmax internally, so the model does not apply softmax before computing its loss. Batch and time are flattened for the loss:

\[
\text{logits}: (B,T,V)\rightarrow(BT,V),\qquad
\text{targets}: (B,T)\rightarrow(BT).
\]

The objective is the average negative log-probability assigned to the correct next character. Backpropagation computes gradients; **AdamW** updates the lookup table after the old gradients are cleared.

Generation takes the last timestep's logits, applies softmax, samples with `torch.multinomial`, and appends the new character. The validation cell reports an actual average over sampled windows. This baseline uses 100 updates and context 8; the later Transformer runs use different training settings.

In [15]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [ ]:
class BigramBaseline(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [16]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [17]:
batch_size = 32
for steps in range(100): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

4.65630578994751


In [18]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


oTo.JUZ!!zqe!
xBP qbs$Gy'AcOmrLwwt
p$x;Seh-onQbfM?OjKbn'NwUAW -Np3fkz$FVwAUEa-wzWC -wQo-R!v -Mj?,SPiTyZ;o-opr$mOiPJEYD-CfigkzD3p3?zvS;ADz;.y?o,ivCuC'zqHxcVT cHA
rT'Fd,SBMZyOslg!NXeF$sBe,juUzLq?w-wzP-h
ERjjxlgJzPbHxf$ q,q,KCDCU fqBOQT
SV&CW:xSVwZv'DG'NSPypDhKStKzC -$hslxIVzoivnp ,ethA:NCCGoi
tN!ljjP3fwJMwNelgUzzPGJlgihJ!d?q.d
pSPYgCuCJrIFtb
jQXg
pA.P LP,SPJi
DBcuBM:CixjJ$Jzkq,OLf3KLQLMGph$O 3DfiPHnXKuHMlyjxEiyZib3FaHV-oJa!zoc'XSP :CKGUhd?lgCOF$;;DTHZMlvvcmZAm;:iv'MMgO&Ywbc;BLCUd&vZINLIzkuTGZa
D.?


In [19]:
m.eval()
with torch.no_grad():
    torch.manual_seed(seed + 10000)
    bigram_val_losses = torch.zeros(20)
    for k in range(20):
        X,Y = get_batch('val')
        logits, loss = m(X,Y)
        bigram_val_losses[k] = loss.item()
m.train()
print('Measured bigram validation loss:', bigram_val_losses.mean().item())

Measured bigram validation loss: 4.586438179016113


## The Mathematical Trick in Self-Attention

### Token Communication and Context Aggregation

Suppose a sequence contains **eight tokens**. The fifth token can receive information from positions 1 through 5, but not positions 6, 7, or 8: those are future tokens relative to its prediction.

The simplest communication rule averages all available representations. At zero-based position \(t\):

\[
\text{xbow}_{b,t}=\frac{1}{t+1}\sum_{j=0}^{t}x_{b,j}.
\]

This represents:

> **the current token + its previous context**

The loop computes this average separately for every batch example and timestep. Both `x` and `xbow` have shape `(B,T,C)`, where **C** is the number of features per token.

### Weighted Aggregation with Matrix Multiplication

`torch.tril` creates a lower-triangular matrix. Dividing each row by its sum assigns equal weights to permitted positions and zero weights to future positions.

For example, `[0.25, 0.25, 0.25, 0.25, 0, 0, 0, 0]` takes 25% of each of the first four representations. Matrix multiplication adds these weighted vectors together.

`wei` has shape `(T,T)` and broadcasts across the batch in `wei @ x`, producing `(B,T,C)`. The toy matrix example makes this weighted sum visible. `torch.allclose` verifies that the matrix version agrees with the loop.

In [20]:
# toy example illustrating how matrix multiplication can be used for a "weighted aggregation"
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [21]:
# consider the following toy example:

torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [22]:
# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)

In [23]:
# version 2: using matrix multiply for a weighted aggregation
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)
assert torch.allclose(xbow, xbow2, atol=1e-6)

### Uniform Weights with Masked Softmax

Every permitted score starts at zero. Future-position scores are replaced with **negative infinity** before softmax:

\[
s_{t,j}=\begin{cases}0 & j\leq t,\\-\infty & j>t.\end{cases}
\]

Softmax exponentiates the scores and divides by their sum. Because \(e^0=1\) and \(e^{-\infty}=0\), each permitted position receives weight \(1/(t+1)\), while future positions receive zero.

`dim=-1` normalizes across source positions within each row. The matrix describes **where a token gathers information from**, rather than mixing different batch examples.

`wei @ x` therefore produces the same average as the loop and normalized triangular matrix. The assertion checks agreement within floating-point tolerance. Learned attention keeps this masking and normalization procedure but replaces the equal scores with data-dependent affinities.

In [24]:
# version 3: use Softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
assert torch.allclose(xbow, xbow3, atol=1e-6)

### The Core Idea of Self-Attention

Uniform averaging treats all previous tokens as **equally important**. Useful context depends on the current token and surrounding text, so self-attention makes communication weights **data-dependent**.

Each token produces three vectors through separate learned transformations:

- **Query (Q):** *What am I looking for?*
- **Key (K):** *What information do I contain / what can I match with?*
- **Value (V):** *What will I communicate if another token finds me relevant?*

Each projection maps `(B,T,C)` to `(B,T,head_size)`. The query at one position is compared with keys at every position:

\[
s_{b,t,j}=q_{b,t}\cdot k_{b,j}.
\]

`q @ k.transpose(-2,-1)` computes `(B,T,H) @ (B,H,T)`, giving `(B,T,T)` affinity scores, where **H** is `head_size`. Stronger alignment produces a larger score.

The causal mask excludes future positions. Softmax converts the remaining scores into weights that sum to one. The model then aggregates **values**, rather than the original input vectors:

\[
o_{b,t}=\sum_{j=0}^{t}a_{b,t,j}v_{b,j}.
\]

The output is `(B,T,H)`. In `wei[0]`, row 4 shows how the fifth token divides its attention among positions 0 through 4. Entries to the right must be zero.

This first demonstration follows the reference's unscaled scores. The next cells explain the scaling used in the final attention implementation.

In [25]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
#out = wei @ x

out.shape

torch.Size([4, 8, 16])

In [26]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

In [27]:
assert torch.allclose(wei.sum(-1), torch.ones(B,T), atol=1e-6)
assert torch.count_nonzero(torch.triu(wei, diagonal=1)) == 0
print('Position 4 attends to positions 0 through 4:', wei[0,4,:5].detach())

Position 4 attends to positions 0 through 4: tensor([0.0294, 0.1052, 0.0469, 0.0276, 0.7909])


### Additional Notes on Attention

#### Positional Information

Attention compares feature vectors; it does not explicitly encode each token's position. The causal mask limits visibility, while **positional embeddings** provide position information:

`x = token_embedding + position_embedding`

Each representation contains information about both **what the token is** and **where it appears in the context**.

---

#### Independent Batches

Each example has its own `(T,T)` attention matrix. Tokens communicate within their sequence; they never communicate with tokens from another batch example.

---

#### Encoder vs. Decoder Attention

The triangular mask creates **causal / decoder-style attention** for next-token prediction. Removing it allows bidirectional communication with earlier and later positions.

In **self-attention**, queries, keys, and values come from the same input. In **cross-attention**, queries come from one source while keys and values come from another, such as an encoder output. This GPT uses causal self-attention.

---

#### Scaled Dot-Product Attention

With independent, approximately unit-variance query and key components, summing H products gives score variance approximately H. The score magnitude therefore grows with head width. Scaling controls this growth:

\[
s_{b,t,j}=\frac{q_{b,t}\cdot k_{b,j}}{\sqrt{H}}.
\]

The random-vector example inspects query, key, and scaled-score variances. The softmax examples show how multiplying scores by eight sharpens the distribution without changing which score is largest.

**Large scores → concentrated softmax → weaker sensitivity for many alternatives.**

**Scaled scores → controlled variance → less premature saturation.**

In [28]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5

In [29]:
k.var()

tensor(1.0449)

In [30]:
q.var()

tensor(1.0700)

In [31]:
wei.var()

tensor(1.0918)

In [32]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [33]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1) # gets too peaky, converges to one-hot

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

## LayerNorm versus Week 6 BatchNorm1d

**Normalization** controls the scale of intermediate representations. BatchNorm and LayerNorm both center activations, divide by a standard deviation, and apply learned scale and shift. Their main difference is **which values contribute to the statistics**.

#### BatchNorm: Statistics Across Examples

Week 6's custom `BatchNorm1d` reduces `(B,C)` over dimension `0`. For `(B,T,C)`, it reduces over `(0,1)`. Each feature channel uses statistics from multiple examples and, in the three-dimensional case, multiple positions.

Training updates `running_mean` and `running_var` with momentum. Evaluation uses these running estimates instead of estimating statistics from the evaluation batch.

#### LayerNorm: Statistics Within One Representation

The educational `LayerNorm1d` below receives `(B,C)` and reduces over dimension `1`. Each example is normalized using its own features. The final `nn.LayerNorm(n_embd)` reduces the last dimension of `(B,T,C)`, independently for each token:

\[
\mu_{b,t}=\frac{1}{C}\sum_{c=1}^{C}x_{b,t,c},\qquad
\sigma^2_{b,t}=\frac{1}{C}\sum_{c=1}^{C}(x_{b,t,c}-\mu_{b,t})^2,
\]

\[
y_{b,t,c}=\gamma_c\frac{x_{b,t,c}-\mu_{b,t}}{\sqrt{\sigma^2_{b,t}+\epsilon}}+\beta_c.
\]

`gamma` and `beta` are learned per-channel parameters. `eps` prevents division by zero. LayerNorm needs **no running mean**: it calculates a token's feature statistics in both training and evaluation, even for one example.

| Property | Week 6 BatchNorm1d | GPT LayerNorm |
|---|---|---|
| Reduction on `(B,T,C)` | batch and time `(0,1)` | channels `-1` |
| Evaluation statistics | running estimates | current token's features |
| Other examples affect training output | yes | no |
| Future positions affect earlier tokens | yes when time is reduced | no |

Reducing across time would let future positions influence an earlier token's normalized representation during training. LayerNorm preserves causal isolation.

#### Read the Following Checks

The custom demonstration uses sample variance, as in the local educational reference. `nn.LayerNorm` uses population variance, so its manual check specifies `unbiased=False`.

`x[:,0]` inspects one feature across examples; `x[0,:]` inspects all features within one example. LayerNorm centers the second group. The final assertion changes other examples and future positions and verifies that earlier tokens' normalized outputs remain unchanged.

In inference, BatchNorm uses its accumulated running statistics, while LayerNorm still uses the current token's features. LayerNorm's formula does not switch between batch estimates and running estimates when the model enters evaluation mode. Its learned scale and shift remain active in both modes.

In [34]:
class LayerNorm1d: # (used to be BatchNorm1d)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

torch.Size([32, 100])

In [35]:
x[:,0].mean(), x[:,0].std() # mean,std of one feature across all batch inputs

(tensor(0.1469), tensor(0.8803))

In [36]:
x[0,:].mean(), x[0,:].std() # mean,std of a single input from the batch, of its features

(tensor(-9.5367e-09), tensor(1.0000))

In [37]:
z = torch.randn(4,8,16)
ln = nn.LayerNorm(16)
zmean = z.mean(-1, keepdim=True)
zvar = z.var(-1, keepdim=True, unbiased=False)
assert torch.allclose(ln(z), (z-zmean)/torch.sqrt(zvar+ln.eps), atol=1e-6)
changed = z.clone()
changed[1:] += 10
changed[0,4:] += 20
assert torch.allclose(ln(z)[0,:4], ln(changed)[0,:4], atol=1e-6)
print('LayerNorm checks passed.')

LayerNorm checks passed.


## Build the Complete GPT

The final model combines **token embeddings, position embeddings, stacked Transformer blocks, final normalization, and a vocabulary projection**.

The reference names the final model `BigramLanguageModel`. Unlike the initial lookup table, it includes attention and stacked Transformer blocks. Its predictions depend on the permitted context through repeated self-attention.

- **`n_embd=64`:** channels in each token representation.
- **`n_head=4`:** parallel heads, each with 16 output channels.
- **`n_layer=4`:** stacked Transformer blocks.
- **`block_size=32`:** maximum visible context and number of position embeddings.
- **`batch_size=16`:** independent windows per update.

The study run shortens training to 200 updates and evaluation to 20 batches; it keeps the 64-channel, four-layer architecture. Tokenization and the chronological split follow the same operations used for the baseline.

In [38]:
# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 200 if study_run else 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 20 if study_run else 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
# ------------

torch.manual_seed(seed)
print('Training updates:', max_iters, 'evaluation batches:', eval_iters)

Training updates: 200 evaluation batches: 20


In [39]:
with open(data_dir / 'input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [40]:
# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,), generator=batch_generator)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

# Keep training-window sampling independent of initialization and dropout.
batch_generator = torch.Generator().manual_seed(seed + 1)

### Estimate Training and Validation Loss

One batch gives a noisy estimate. `estimate_loss()` averages losses over several equally sized batches from each split:

\[
\widehat{L}_{\text{split}}=\frac{1}{K}\sum_{k=1}^{K}L_k.
\]

Each batch contains the same number of targets, so this gives equal weighting to sampled positions. It is not a full-corpus loss.

`@torch.no_grad()` prevents evaluation from building gradient graphs. `model.eval()` disables dropout; it does not itself disable gradients. The previous model mode is restored afterward.

Resetting the temporary generator gives matching configurations identical evaluation windows. Restoring `training_generator` prevents evaluation from advancing the sequence of training windows.

Cross-entropy is measured in **natural-log units**. Lower loss means greater average probability assigned to the observed next characters.

In [41]:
@torch.no_grad()
def estimate_loss():
    global batch_generator
    out = {}
    was_training = model.training
    training_generator = batch_generator
    model.eval()
    try:
        for split in ['train', 'val']:
            batch_generator = torch.Generator().manual_seed(seed + 10000 + (split == 'val'))
            losses = torch.zeros(eval_iters)
            for k in range(eval_iters):
                X, Y = get_batch(split)
                logits, loss = model(X, Y)
                losses[k] = loss.item()
            out[split] = losses.mean()
        return out
    finally:
        batch_generator = training_generator
        model.train(was_training)

### One Head of Self-Attention

`Head` packages the earlier query-key-value operations into an `nn.Module`. Each bias-free projection maps `n_embd` to `head_size` channels.

The triangular mask is a registered **buffer**: it moves with the model but is not learned. `self.tril[:T,:T]` supports shorter contexts, including the first steps of generation.

**input → queries and keys → scaled scores → causal mask → softmax → dropout → weighted values**

Weights are `(B,T,T)` and values are `(B,T,H)`, giving output `(B,T,H)`. The scale `k.shape[-1]**-0.5` uses the actual head width. It corrects the local finished reference's input-width scaling and matches Week 7's explanation.

Dropout acts after softmax: training removes individual weights and rescales retained weights. A dropped row need not sum to one. Evaluation disables dropout and uses the original normalized weights.

In [42]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,head_size)
        q = self.query(x) # (B,T,head_size)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B,T,head_size) @ (B,head_size,T) -> (B,T,T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,head_size)
        out = wei @ v # (B,T,T) @ (B,T,head_size) -> (B,T,head_size)
        return out

# 1. Tek head'i multi-head attention'a çevir: aynı boyutu n_head tane küçük head'e böl, çıktıları birleştir. Val loss'u geçen haftaki tek head'le yan yana koy.

## 1. Multiple Heads in Parallel

A single head computes one communication pattern. **Multi-head attention** lets independent learned projections gather different information from the same input.

The total output width is divided between heads:

\[
H=\frac{C}{n_{\text{head}}}.
\]

With 64 channels and four heads, each output is `(B,T,16)`. Concatenation along `dim=-1` gives:

**four `(B,T,16)` outputs → concatenate → `(B,T,64)`**

Every head reads the full input vector; its **projected output width** is smaller. Heads have separate parameters and attention patterns, rather than being assigned fixed slices of the input.

The output projection mixes the concatenated channels. Dropout follows that projection. Preserving total width also enables residual additions later.

The component comparison below measures single-head and multi-head models at the same total width, with the same projection and training settings. **The recorded single-head loss is a newly trained Week 8 baseline, not a historical Week 7 result.** Week 7 ends with the instruction to integrate a head into a language model; its code demonstrates queries, keys, and values on random tensors, but contains no trained single-head language model or saved single-head validation loss.

The Week 7 toy uses input width 32, head width 16, and an unscaled attention demonstration. The Week 8 baseline uses width 32, one 32-channel scaled head, positional embeddings, an output projection, and context 32. It implements the requested next step, but is not an exact reproduction of a trained Week 7 model. To obtain a genuine historical comparison, a trained Week 7 model or checkpoint and its configuration would be needed. Training a model now would create a new measurement.

In [43]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

# 2. Transformer bloğunu parça parça tamamla: feedforward, residual bağlantı, LayerNorm. Her parçayı ekledikten sonra val loss'u kaydet. Tek tablo çıkar: bigram, tek head, multi-head, feedforward eklenmiş, residual eklenmiş, LayerNorm eklenmiş. LayerNorm'u hafta 6'da yazdığın BatchNorm1d ile karşılaştır: hangi eksende normalize ediyor, neden running_mean tutmuyor, kendi cümlelerinle yaz.

## 2. Complete the Transformer Block

### Feedforward Computation

Attention lets tokens **communicate**. The feedforward network then lets each token **process** its updated representation. The same network is applied independently at every position; it does not mix the time dimension.

\[
\operatorname{FFN}(x)=\operatorname{ReLU}(xW_1+b_1)W_2+b_2.
\]

The first linear layer expands C channels to 4C. ReLU introduces a nonlinearity; the second layer projects back to C:

**`(B,T,C)` → `(B,T,4C)` → ReLU → `(B,T,C)` → dropout**

Expansion provides a larger intermediate feature space while preserving the output shape needed by the block. Without the nonlinearity, the two linear transformations could be combined into one affine transformation.

`FeedFoward` is the class name used in the reference implementation. The complete block normalizes each token before attention and feedforward, using the feature-axis normalization explained in the Week 6 comparison.

Attention combines information through weighted value sums. The feedforward network adds nonlinear feature processing after that combination, allowing the model to transform what it has gathered. Attention is the communication step; feedforward is the token-wise computation step.

In [44]:
class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

### Residual Connections and Pre-Normalized Blocks

Repeatedly replacing a representation can make it harder for information and gradients to pass through a deep stack. A **residual connection** adds a sublayer's output to its input:

\[
y=x+f(x).
\]

Both terms must be `(B,T,C)`, which explains the attention projection and feedforward output width. The model learns an update to the existing representation. Backpropagation has an identity path in addition to the sublayer derivative.

The complete block uses **pre-normalization**:

\[
x'=x+\operatorname{Attention}(\operatorname{LayerNorm}_1(x)),
\]

\[
x''=x'+\operatorname{FFN}(\operatorname{LayerNorm}_2(x')).
\]

`ln1` and `ln2` have separate learned scale and shift parameters. Normalization controls each sublayer input, while the residual path carries the existing representation forward.

The following block therefore performs **communication followed by computation**, with normalized inputs and residual connections around both operations.

As more blocks are stacked, this direct path becomes more useful: gradients need not pass only through a chain of attention and nonlinear transformations. Residual connections help optimization, but do not guarantee that every gradient remains well scaled.

In [45]:
class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

### Token Embeddings, Position Embeddings, and the Vocabulary Head

`token_embedding_table` maps `(B,T)` IDs to `(B,T,C)` vectors. `position_embedding_table` maps positions to `(T,C)` vectors, which broadcast across the batch when added to token embeddings.

**character identity + context position → token representation**

`self.blocks` applies `n_layer` blocks. Each preserves `(B,T,C)` while gathering and processing context. Final LayerNorm precedes `lm_head`, which maps C features to V next-character logits.

With targets, this implementation returns flattened `(B*T,V)` logits after computing cross-entropy. Without targets, it returns `(B,T,V)` for generation.

#### Autoregressive Generation

At each iteration, the model:

1. Crops the sequence to its last `block_size` characters.
2. Computes logits for that context.
3. Selects the last timestep's `(B,V)` logits.
4. Applies softmax and samples one character.
5. Appends the sampled ID to the growing sequence.

Cropping is necessary because the position table and mask have fixed maximum sizes. The full generated sequence grows; only the context supplied for prediction is cropped. Sampling is stochastic rather than an argmax choice.

The surrounding generation cells use evaluation mode and `torch.no_grad()`. The checks after model construction verify flattened logits, finite gradients, causal isolation, and generation beyond the context length.

In [46]:
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [47]:
model = BigramLanguageModel()
m = model.to(device)
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

0.209729 M parameters


In [48]:
# lightweight checks before training
xb, yb = get_batch('train')
logits, loss = model(xb, yb)
assert logits.shape == (batch_size*block_size, vocab_size)
assert torch.isfinite(loss)
loss.backward()
assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())
optimizer.zero_grad(set_to_none=True)
model.eval()
changed = xb.clone()
changed[:,4:] = (changed[:,4:] + 1) % vocab_size
with torch.no_grad():
    assert torch.allclose(model(xb)[0][:,:4], model(changed)[0][:,:4], atol=1e-6)
    generated = model.generate(xb[:1,:1], max_new_tokens=block_size+2)
    assert generated.shape == (1,block_size+3)
model.train()
print('Shape, gradient, causal mask, and context-cropping checks passed.')

Shape, gradient, causal mask, and context-cropping checks passed.


### Train and Generate

Each update samples a fresh batch and computes cross-entropy through `model(xb,yb)`. The update follows:

**clear old gradients → backpropagate the loss → update parameters**

`zero_grad(set_to_none=True)` prevents accumulation across updates. `loss.backward()` differentiates through embeddings, attention, feedforward, and normalization. `optimizer.step()` applies AdamW's update.

Periodic evaluation measures train and validation loss without updating parameters. The last periodic report occurs before its optimizer update; the final separate evaluation measures the model after all updates.

CUDA synchronization ensures pending device work finishes before the timer is read. Duration includes periodic evaluation and excludes generation, so it measures this training workflow rather than optimizer computation alone.

The generation cell sets the seed, uses evaluation mode, and prints actual samples. Short training can reveal character patterns without producing coherent Shakespeare.

In [49]:
if device == 'cuda':
    torch.cuda.synchronize()
start = time.perf_counter()

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    assert torch.isfinite(loss)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

if device == 'cuda':
    torch.cuda.synchronize()
training_seconds = time.perf_counter() - start
losses = estimate_loss()
print('Final train loss:', losses['train'].item(), 'final val loss:', losses['val'].item())

step 0: train loss 4.4142, val loss 4.4076

step 100: train loss 2.6511, val loss 2.6518


step 199: train loss 2.4845, val loss 2.4945


Final train loss: 2.4877538681030273 final val loss: 2.496487617492676


In [50]:
# generate from the model
model.eval()
torch.manual_seed(seed + 2)
context = torch.zeros((1, 1), dtype=torch.long, device=device)
with torch.no_grad():
    english_sample = decode(m.generate(context, max_new_tokens=250 if study_run else 2000)[0].tolist())
print(english_sample)
results = [{'model':'reference GPT', 'parameters':sum(p.numel() for p in model.parameters()),
            'seconds':training_seconds, 'train_loss':losses['train'].item(),
            'val_loss':losses['val'].item(), 'updates':max_iters}]
reference_block = Block
reference_model = BigramLanguageModel
english_architecture = (n_embd, n_head, n_layer, block_size, dropout)


MH lan wherou?
A, nomare poto ha:
A'trerilan wch,
BHark th
Sopownd housoourome, hy me bre,

Tha cen derd h ra tousurray tld 't ollee, g ar lin alin iss telorefhenm?


Dy teerealo ngse st w m y m telpowirmiesceite La, y ve row
IRemend mel kghe maswher


### Compare the Incremental Transformer Components

The complete block above contains all components. The following cells explicitly implement the earlier stages:

- **AttentionBlock:** attention alone.
- **FeedForwardBlock:** attention followed by per-token computation.
- **ResidualBlock:** residual paths around both operations.
- **The complete Block:** pre-normalized residual attention and feedforward.

These stages reuse the same heads, projections, embeddings, optimizer, and loss estimator. `AttentionLanguageModel` uses `nn.Identity()` instead of final normalization; the complete model restores it.

The comparison starts with one head and then four heads. Both use 32 total channels, one block, context 32, batch 16, and the same update budget. Their parameter counts are equal. Feedforward adds parameters, residual additions do not, and LayerNorm adds learned scales and shifts.

The fixed seeds and evaluation windows support reproducibility within this environment. A short single-seed comparison measures optimization behavior, not eventual superiority.

In [51]:
class AttentionBlock(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)

    def forward(self, x):
        return self.sa(x)

class FeedForwardBlock(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)

    def forward(self, x):
        x = self.sa(x)
        x = self.ffwd(x)
        return x

class ResidualBlock(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)

    def forward(self, x):
        x = x + self.sa(x)
        x = x + self.ffwd(x)
        return x

In [52]:
class AttentionLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.Identity() # final LayerNorm is introduced with the normalized block
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

### Keep the Training Workflow the Same

`train_and_record` creates a fresh model and optimizer, resets initialization and training-window sampling, and repeats the same training operations used above.

Rows enter `results` only after training and final evaluation finish. Samples come from those trained models. Selecting `Block` and `BigramLanguageModel` before a call chooses the explicit implementation; it does not reuse the preceding model's learned weights.

In [53]:
def train_and_record(name):
    global model, m, optimizer, batch_generator
    torch.manual_seed(seed)
    batch_generator = torch.Generator().manual_seed(seed + 1)
    model = BigramLanguageModel()
    m = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    if device == 'cuda':
        torch.cuda.synchronize()
    start = time.perf_counter()

    for iter in range(max_iters):

        # every once in a while evaluate the loss on train and val sets
        if iter % eval_interval == 0 or iter == max_iters - 1:
            losses = estimate_loss()
            print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

        # sample a batch of data
        xb, yb = get_batch('train')

        # evaluate the loss
        logits, loss = model(xb, yb)
        assert torch.isfinite(loss)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    if device == 'cuda':
        torch.cuda.synchronize()
    training_seconds = time.perf_counter() - start
    losses = estimate_loss()
    print('Final train loss:', losses['train'].item(), 'final val loss:', losses['val'].item())
    results.append({'model':name, 'parameters':sum(p.numel() for p in model.parameters()),
                    'seconds':training_seconds, 'train_loss':losses['train'].item(),
                    'val_loss':losses['val'].item(), 'updates':max_iters})
    model.eval()
    torch.manual_seed(seed + 2)
    context = torch.zeros((1,1), dtype=torch.long, device=device)
    with torch.no_grad():
        sample = decode(m.generate(context, max_new_tokens=250 if study_run else 2000)[0].tolist())
    return sample

### Single-Head Validation Run

With `n_embd=32` and `n_head=1`, one head produces all 32 output channels. The output projection is retained. Feedforward, residual paths, and normalization are absent at this stage.

In [54]:
n_embd = 32
n_layer = 1
n_head = 1
dropout = 0.0
Block = AttentionBlock
BigramLanguageModel = AttentionLanguageModel
single_head_sample = train_and_record('single head')

step 0: train loss 4.1894, val loss 4.1905


step 100: train loss 3.1962, val loss 3.2269


step 199: train loss 3.0462, val loss 3.0577
Final train loss: 3.0442843437194824 final val loss: 3.055230140686035


### Multi-Head Validation Run

Only the number of heads changes to four. Each now produces eight channels; concatenation restores 32. Compare the measured loss with the preceding single-head run under the same settings.

In [55]:
n_head = 4
multi_head_sample = train_and_record('multi-head')

step 0: train loss 4.2013, val loss 4.2005


step 100: train loss 3.1401, val loss 3.1777


step 199: train loss 2.9156, val loss 2.9300
Final train loss: 2.9135403633117676 final val loss: 2.9276528358459473


### Add Feedforward Computation

`FeedForwardBlock` processes attention output through the expanded ReLU network. It still replaces the representation rather than adding a residual update. Additional parameters change capacity as well as computation.

In [56]:
Block = FeedForwardBlock
feedforward_sample = train_and_record('+ feedforward')

step 0: train loss 4.1537, val loss 4.1557


step 100: train loss 3.1426, val loss 3.1721


step 199: train loss 2.8796, val loss 2.8859
Final train loss: 2.877540111541748 final val loss: 2.883774995803833


### Add Residual Connections

`ResidualBlock` retains attention and feedforward but adds each output back to its input. Parameter count is unchanged; the information and gradient paths differ.

In [57]:
Block = ResidualBlock
residual_sample = train_and_record('+ residual')

step 0: train loss 4.4608, val loss 4.4705


step 100: train loss 2.9366, val loss 2.9665


step 199: train loss 2.7285, val loss 2.7564
Final train loss: 2.7272846698760986 final val loss: 2.7546987533569336


### Add LayerNorm

Restoring the complete `Block` adds separate pre-normalizations for attention and feedforward. The complete language model also restores final LayerNorm before the vocabulary projection. The run measures their combined effect.

In [58]:
Block = reference_block
BigramLanguageModel = reference_model
normalized_sample = train_and_record('+ LayerNorm')

step 0: train loss 4.2905, val loss 4.2980


step 100: train loss 3.0590, val loss 3.0876


step 199: train loss 2.7755, val loss 2.8068
Final train loss: 2.773606061935425 final val loss: 2.8048453330993652


In [59]:
def show_results(rows):
    lines = ['| Model | Parameters | Updates | Training seconds | Train loss | Validation loss |',
             '|---|---:|---:|---:|---:|---:|']
    for row in rows:
        lines.append(f"| {row['model']} | {row['parameters']:,} | {row['updates']} | {row['seconds']:.2f} | {row['train_loss']:.4f} | {row['val_loss']:.4f} |")
    display(Markdown('\n'.join(lines)))

show_results(results[1:])

| Model | Parameters | Updates | Training seconds | Train loss | Validation loss |
|---|---:|---:|---:|---:|---:|
| single head | 9,377 | 200 | 0.69 | 3.0443 | 3.0552 |
| multi-head | 9,377 | 200 | 0.95 | 2.9135 | 2.9277 |
| + feedforward | 17,729 | 200 | 1.07 | 2.8775 | 2.8838 |
| + residual | 17,729 | 200 | 1.26 | 2.7273 | 2.7547 |
| + LayerNorm | 17,921 | 200 | 1.09 | 2.7736 | 2.8048 |

### Read the Saved Component Comparison

These are the original saved validation measurements. The bigram row is an earlier baseline; the attention rows are Week 8 runs.

| Model | Saved validation loss |
|---|---:|
| Bigram | 4.5864 |
| Single head (Week 8 baseline) | 3.0552 |
| multi-head | 2.9277 |
| + feedforward | 2.8838 |
| + residual | 2.7547 |
| + LayerNorm | 2.8048 |

The bigram uses context 8, batch 32, and 100 updates. Component runs use context 32, batch 16, and 200 updates; evaluation windows also differ. The baseline is recorded here, but its difference cannot be interpreted as an architecture-only comparison.

The normalized component model has higher saved validation loss than the residual-only model. This short-run result does not establish their eventual ranking after convergence.

### Which Settings Match?

| Setting | Earlier bigram | Week 8 component runs |
|---|---|---|
| Batch size | 32 | 16 |
| Context length | 8 | 32 |
| Optimizer updates | 100 | 200 |
| Split | chronological 90/10, English corpus | same rule and corpus |
| Evaluation batches | 20 validation batches | 20 batches per split |
| Initialization seed | 1337 | 1337 for each run |
| Training windows | shared global RNG, also used by generation | independent generator, seed 1338 |
| Validation windows | global RNG reset to 11337 | separate generator reset to 11338 |
| Optimizer | AdamW, learning rate 0.001 | same |
| AdamW defaults | betas (0.9, 0.999), eps 1e-8, weight decay 0.01 | same |

The earlier bigram sees 25,600 target positions during training; each component run sees 102,400. This is a fourfold difference in token budget. Bigram predictions do not use earlier context, but context length still changes the number and grouping of training targets. Identical seed numbers also do not imply identical sampled windows when the RNG is shared with initialization or generation.

Within the five component runs, data windows and optimization settings match. Single-head and multi-head models also have equal parameter counts. Feedforward and LayerNorm add parameters, so those comparisons measure the component together with its capacity change. The LayerNorm stage adds both sublayer normalization and final normalization.

### What Do the Short Runs Show?

The saved final train/validation losses are 3.0443/3.0552 for single-head, 2.9135/2.9277 for multi-head, 2.8775/2.8838 with feedforward, 2.7273/2.7547 with residuals, and 2.7736/2.8048 with LayerNorm. These small gaps show no clear sign of strong overfitting at the recorded endpoint. They do not rule out later overfitting or variation between sampled windows.

Training and validation loss both decrease between the recorded evaluations. The higher normalized-model loss after 200 updates describes early optimization, not proof that LayerNorm lowers final model quality. Longer runs with repeated seeds and learning curves are needed before drawing that conclusion. The later opt-in protocol prepares matching bigram and component runs without replacing this table.

# 3. Modeli büyüt ve dropout ekle. Elindeki donanıma sığan bir konfigürasyon seç: n_embd, n_head, n_layer, block_size. Parametre sayısını, val loss'u ve eğitim süresini yaz, neden bu değerleri seçtiğini açıkla. Modelin ürettiği metinden bir örnek koy.

## 3. GPT Scaling and Dropout

Greater **width** gives tokens larger representations. Greater **depth** provides repeated communication and computation. Neither guarantees better validation loss after a short training budget, so the table reports measured results.

The small GPT uses 32 channels, four heads, and one layer. The reference GPT uses 64 channels, four heads, and four layers. Both have context 32 and batch 16. These modest context and batch sizes keep attention matrices and training activations manageable on the available GPU.

Each head creates `(B,T,T)` scores, so attention-score storage grows quadratically with context length. Linear-layer parameter counts grow approximately quadratically with width; stacking more blocks grows parameters approximately linearly with depth. These dimensions therefore affect memory in different ways.

#### Why Add Dropout?

**Dropout** randomly removes activations during training. Retained activations are scaled by \(1/(1-p)\) for drop probability p, preserving their expected value. Evaluation disables the random removal.

Here dropout acts on attention weights, projected attention output, and feedforward output. The dropout run uses probability 0.2 and otherwise matches the reference-sized GPT. It adds no learned parameters.

The table reports **parameter count, validation loss, and training duration**. Runs share the update/token budget rather than the duration. The printed samples are actual saved generation outputs; 200 updates do not establish convergence.

Dropout is a **regularization technique**: random removal discourages reliance on particular activation paths. It can increase training difficulty and may need more updates before any validation benefit appears. Here the no-dropout reference model ends at train/validation 2.4878/2.4965, while dropout ends at 2.5148/2.5234. Their small gaps do not establish strong overfitting or a benefit from dropout in this short run.

The initial reference GPT also underwent forward/backward and generation checks before training. Those checks advanced its training-window generator and global sampling RNG. The dropout and component runs reset their generators immediately before training. The saved reference/dropout comparison therefore does not perfectly control training windows, despite equal nominal settings. A fresh matched comparison should reset RNG state after checks; the opt-in protocol below does that for every model.

In [60]:
n_embd, n_head, n_layer, block_size, dropout = english_architecture
dropout = 0.2
dropout_sample = train_and_record('reference GPT + dropout')
show_results([results[5], results[0], results[6]])
for name, sample in [('small GPT', normalized_sample), ('reference GPT', english_sample),
                     ('reference GPT + dropout', dropout_sample)]:
    print('\nGenerated Shakespeare text:', name)
    print(sample)

step 0: train loss 4.4142, val loss 4.4076


step 100: train loss 2.6750, val loss 2.6732


step 199: train loss 2.5169, val loss 2.5260


Final train loss: 2.5148377418518066 final val loss: 2.523376941680908


| Model | Parameters | Updates | Training seconds | Train loss | Validation loss |
|---|---:|---:|---:|---:|---:|
| + LayerNorm | 17,921 | 200 | 1.09 | 2.7736 | 2.8048 |
| reference GPT | 209,729 | 200 | 3.70 | 2.4878 | 2.4965 |
| reference GPT + dropout | 209,729 | 200 | 4.65 | 2.5148 | 2.5234 |


Generated Shakespeare text: small GPT

MH lhe wh bou?Ig,$nohore p toTe :
A'trerilan rcdhe wrdk th

gposndinou oouNome,$hy me b E


Ty  cv W
We on ra tousurlher ldB't ollee w, a

lia titT ith telooNfhenm?




T terealo WLs

T pe d y m telpos,Bmimec'ite L , y vu rou

Re h:
UTrarkghe maswthy

Generated Shakespeare text: reference GPT

MH lan wherou?
A, nomare poto ha:
A'trerilan wch,
BHark th
Sopownd housoourome, hy me bre,

Tha cen derd h ra tousurray tld 't ollee, g ar lin alin iss telorefhenm?


Dy teerealo ngse st w m y m telpowirmiesceite La, y ve row
IRemend mel kghe maswher

Generated Shakespeare text: reference GPT + dropout

MH lan wherou?
A, inhare poto haro harerilan wch,
Bcrdk th
Sopound housoourome, hy m, boE,

Tha ce bour on ra tousurray, ldB't ollee, g ar lin alin iss teloo fhenme,

Dy ther alo ngs

The, m y m telpowirmiesceite La, y ve row

Re hed hinokghe maswher


# 4. Aynı modeli Türkçe metinle eğit. Metni kendin derle, Tiny Shakespeare'e yakın boyutta olsun (yaklaşık 1 MB). Kaynaklardaki Vikikaynak'tan başlayabilirsin. Ürettiği metinden örnek ver. Türkçe metnin kaç farklı karakteri var, Shakespeare'inkiyle karşılaştır. İki val loss'u doğrudan karşılaştırabilir misin, neden, yaz.

## 4. Turkish Character-Level Modeling

The Turkish tokenizer repeats the same operations as the English tokenizer. UTF-8 reading without lowercasing preserves distinctions such as **`i`, `ı`, `İ`, and `I`**. Spaces, punctuation, accented letters, and newlines also receive separate IDs when present.

`Counter(text)` displays character frequencies. `sorted(set(text))` gives the vocabulary: the supplied Turkish corpus has **92 unique characters**, compared with **65** in Shakespeare. Vocabulary changes affect the embedding and output-layer dimensions, not the attention mechanism.

The exact English reference configuration is restored: 64 channels, four heads, four layers, context 32, and zero dropout. Only the corpus, token mappings, vocabulary size, and dataset tensors change. The chronological 90/10 split and training settings remain the same.

The existing Turkish file has 302,802 characters and is smaller than the assignment's approximate 1 MB target. This experiment documents the supplied file; it does not establish that the collection and size requirements have been met.

#### Can the Validation Losses Be Compared Directly?

Character cross-entropy measures:

\[
L=-\frac{1}{BT}\sum_{b,t}\log p(y_{b,t}\mid x_{b,\leq t}).
\]

Both losses have the same unit, but the prediction problems differ. Vocabulary, character frequencies, punctuation, genre, and predictability affect the loss. Even a uniform predictor has loss \(\log V\), which depends on vocabulary size.

The values can be reported side by side, but lower Turkish loss alone does not establish better Turkish modeling or lower language difficulty. Read the generated example as an actual sample from this corpus and training budget.

In [61]:
with open(data_dir / 'turkish_input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print('length of dataset in characters:', len(text))
print('vocabulary size:', vocab_size)
print('vocabulary:', repr(''.join(chars)))
counts = Counter(text)
print('most frequent characters:', [(repr(ch), count) for ch,count in counts.most_common(20)])
for ch in 'çğıöşüÇĞİÖŞÜiI':
    print(repr(ch), 'count:', counts[ch], 'token:', stoi.get(ch))
assert decode(encode(text[:200])) == text[:200]
print(text[:400])

length of dataset in characters: 302802
vocabulary size: 92
vocabulary: "\n !'()*,-.012349:;?ABCDEFGHIJKLMNOPRSTUVWYZabcdefghijklmnoprstuvwyz«\xad»ÂÇÎÖÜâçéîöûüğİıŞşʾ\u2003’“”"
most frequent characters: [("' '", 39009), ("'a'", 28457), ("'e'", 22803), ("'i'", 21158), ("'n'", 17167), ("'r'", 16937), ("'d'", 12678), ("'l'", 12247), ("'ı'", 11515), ("'m'", 11332), ("'k'", 11213), ("'t'", 8439), ("'u'", 8283), ("'y'", 7451), ("'b'", 7367), ("'s'", 6404), ("'o'", 5525), ("'.'", 5060), ("'ü'", 4890), ("'ş'", 4158)]
'ç' count: 2526 token: 76
'ğ' count: 2846 token: 82
'ı' count: 11515 token: 84
'ö' count: 1503 token: 79
'ş' count: 4158 token: 86
'ü' count: 4890 token: 81
'Ç' count: 54 token: 71
'Ğ' count: 0 token: None
'İ' count: 183 token: 83
'Ö' count: 30 token: 73
'Ş' count: 75 token: 85
'Ü' count: 12 token: 74
'i' count: 21158 token: 51
'I' count: 4 token: 27
 Simdiye kadar tesadüf ettiğim insanlardan bir tanesi benim üzerimde belki en büyük tesiri yapmıştır. Aradan aylar geçtiği halde bir 

In [62]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]
n_embd, n_head, n_layer, block_size, dropout = english_architecture
assert (n_embd, n_head, n_layer, block_size, dropout) == english_architecture
turkish_sample = train_and_record('Turkish reference GPT')
show_results([results[0], results[-1]])
print('Generated Turkish text:')
print(turkish_sample)

step 0: train loss 4.7596, val loss 4.7384


step 100: train loss 2.6710, val loss 2.6800


step 199: train loss 2.4872, val loss 2.4689


Final train loss: 2.4849016666412354 final val loss: 2.467500686645508


| Model | Parameters | Updates | Training seconds | Train loss | Validation loss |
|---|---:|---:|---:|---:|---:|
| reference GPT | 209,729 | 200 | 3.70 | 2.4878 | 2.4965 |
| Turkish reference GPT | 213,212 | 200 | 4.05 | 2.4849 | 2.4675 |

Generated Turkish text:

 Bâhda ta ak sıcamıydünimü bk stan yananedatm Yidirınniş biz lüz gedirikirikinundi ytim ktıbi yaysa kakt danı tammunndan,çamı kkı şa süramiheğiyekiredi yahekabde,­neyaladinak ihk k or ora ri valınahldordinatı­, a Fin vçst yatımamidırdünıngcak in tu s


The saved measurements use sampled windows rather than every validation position. They document completed runs, including their different baseline settings. Additional measurements should be stored separately so that the original losses, durations, and generated text remain available.

In [63]:
print('Completed Transformer experiments:', len(results))
print('Updates per Transformer experiment:', max_iters)
if study_run:
    print('The 5,000-update training budget has not been executed in this run.')
print('Repeated seeds and full-corpus validation have not been executed.')

Completed Transformer experiments: 8
Updates per Transformer experiment: 200
The 5,000-update training budget has not been executed in this run.
Repeated seeds and full-corpus validation have not been executed.


### Prepare a Matched Comparison

This optional workflow uses the existing bigram implementation, attention stages, batching function, and `estimate_loss()`. It is defined below but is **not run automatically**.

Each model starts fresh on English text with batch 16, context 32, 200 updates, 20 evaluation batches, seed 1337, training-window seed 1338, and AdamW at learning rate 0.001. Optimizer defaults are specified explicitly. Matching settings makes the bigram and component token budgets equal; it does not make their capacities or visible information identical.

The returned rows have their own labels and include the configuration and dataset hash. The function restores the surrounding dataset, model bindings, generators, and RNG state. It does not append to `results` or overwrite existing tables. This is a new Week 8 single-head baseline, never a recovered Week 7 result.

To run the short comparison deliberately, call `matched_results = run_matched_comparison()` and then `show_results(matched_results)`. To prepare a longer run, the same function accepts `updates=5000` and `evaluation_batches=200`. That trains six fresh models; obtain approval before starting the longer run. No matched-protocol or longer-training losses have been measured here.

In [2]:
def run_matched_comparison(updates=200, evaluation_batches=20, experiment_seed=1337):
    global model, m, optimizer, batch_generator, train_data, val_data
    global vocab_size, n_embd, n_head, n_layer, block_size, batch_size
    global dropout, seed, eval_iters, Block, BigramLanguageModel
    assert updates > 0 and evaluation_batches > 0
    names = ['model', 'm', 'optimizer', 'batch_generator', 'train_data', 'val_data',
             'vocab_size', 'n_embd', 'n_head', 'n_layer', 'block_size', 'batch_size',
             'dropout', 'seed', 'eval_iters', 'Block', 'BigramLanguageModel']
    saved = {name: globals()[name] for name in names}
    matched_rows = []
    english_text = (data_dir / 'input.txt').read_text(encoding='utf-8')
    english_stoi = {ch:i for i,ch in enumerate(sorted(set(english_text)))}
    english_data = torch.tensor([english_stoi[ch] for ch in english_text], dtype=torch.long)
    split = int(0.9 * len(english_data))
    import hashlib
    dataset_hash = hashlib.sha256((data_dir / 'input.txt').read_bytes()).hexdigest()
    stages = [('Bigram (matched Week 8)', None),
              ('Single head (new Week 8 baseline)', AttentionBlock),
              ('Multi-head (matched)', AttentionBlock),
              ('+ feedforward (matched)', FeedForwardBlock),
              ('+ residual (matched)', ResidualBlock),
              ('+ LayerNorm (matched)', reference_block)]
    try:
        devices = [torch.cuda.current_device()] if device == 'cuda' else []
        with torch.random.fork_rng(devices=devices):
            train_data, val_data = english_data[:split], english_data[split:]
            vocab_size = len(english_stoi)
            n_embd, n_layer, block_size, batch_size, dropout = 32, 1, 32, 16, 0.0
            seed, eval_iters = experiment_seed, evaluation_batches
            for index, (name, block_class) in enumerate(stages):
                torch.manual_seed(seed)
                n_head = 1 if index == 1 else 4
                if block_class is None:
                    model = BigramBaseline(vocab_size).to(device)
                else:
                    Block = block_class
                    BigramLanguageModel = reference_model if index == 5 else AttentionLanguageModel
                    model = BigramLanguageModel().to(device)
                m = model
                optimizer = torch.optim.AdamW(model.parameters(), lr=0.001,
                    betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01)
                batch_generator = torch.Generator().manual_seed(seed + 1)
                initial = estimate_loss()
                model.train()
                if device == 'cuda':
                    torch.cuda.synchronize()
                start = time.perf_counter()
                for step in range(updates):
                    xb, yb = get_batch('train')
                    logits, loss = model(xb, yb)
                    assert torch.isfinite(loss)
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    optimizer.step()
                if device == 'cuda':
                    torch.cuda.synchronize()
                duration = time.perf_counter() - start
                final = estimate_loss()
                matched_rows.append({'model':name,
                    'parameters':sum(p.numel() for p in model.parameters()),
                    'seconds':duration, 'updates':updates,
                    'train_loss':final['train'].item(), 'val_loss':final['val'].item(),
                    'initial_val_loss':initial['val'].item(),
                    'batch_size':batch_size, 'block_size':block_size,
                    'eval_iters':eval_iters, 'seed':seed,
                    'dataset_sha256':dataset_hash,
                    'duration_scope':'optimizer loop only; excludes evaluation'})
        return matched_rows
    finally:
        globals().update(saved)

Executed: matched_results = run_matched_comparison()
Settings: 200 updates per model; batch 16; context 32; 20 evaluation batches; seed 1337.
Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


| Model | Parameters | Updates | Training seconds | Train loss | Validation loss |
|---|---:|---:|---:|---:|---:|
| Bigram (matched Week 8) | 4,225 | 200 | 0.41 | 4.4754 | 4.4745 |
| Single head (new Week 8 baseline) | 9,377 | 200 | 0.47 | 3.0443 | 3.0552 |
| Multi-head (matched) | 9,377 | 200 | 0.90 | 2.9135 | 2.9277 |
| + feedforward (matched) | 17,729 | 200 | 0.88 | 2.8775 | 2.8838 |
| + residual (matched) | 17,729 | 200 | 0.87 | 2.7273 | 2.7547 |
| + LayerNorm (matched) | 17,921 | 200 | 1.00 | 2.7736 | 2.8048 |

Total matched-comparison execution time: 7.34 seconds (including evaluation and table display).
All six experiments completed successfully.
